In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# setup
from src.read_data.read_data import read_data
from src.read_data.translate_names import translate_to_english
from src.read_data.fix_types import fix_data_types
from src.data_cleaning.feature_engineering import identify_unemployment_episodes
from src.data_cleaning.feature_engineering import generate_biographical_variables
from src.data_cleaning.feature_engineering import index_unemployment_entries
from src.data_cleaning.feature_engineering import generate_employment_history
from src.data_cleaning.transform import reshape_data_for_ml
from src.data_cleaning.missing_values import replace_missing_value_placeholder
from src.data_cleaning.missing_values import handle_missing_values
from src.modeling.setup import select_features, select_target
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", None)

In [ ]:
df = read_data(500000)
df = translate_to_english(df)
df = fix_data_types(df)
df = identify_unemployment_episodes(df)
df = generate_biographical_variables(df)
df = index_unemployment_entries(df)
df = generate_employment_history(df, 1)
df = reshape_data_for_ml(df)
df = replace_missing_value_placeholder(df)

In [ ]:
# impute NaNs in different ways
df_drop_cols = handle_missing_values(df, method="drop_columns")
df_impute_median = handle_missing_values(df, method="median")
# need to drop any features that have only NaNs
df_impute_median = handle_missing_values(df_impute_median, method="drop_columns")

## Modeling

With the data prepared, we can now move on to modeling. This involves selecting features and target variables, splitting the data into training and testing sets, and training a machine learning model.

In [ ]:
X_drop_cols = select_features(df_drop_cols)
X_impute_median = select_features(df_impute_median)
# remove ue_duration and unemployment_duration_days features
X_drop_cols = X_drop_cols.drop(columns=["ue_duration", "unemployment_duration_days"])
X_impute_median = X_impute_median.drop(
    columns=["ue_duration", "unemployment_duration_days"]
)
y = select_target(df)

In [ ]:
X_cols_train, X_cols_test, X_median_train, X_median_test, y_train, y_test = (
    train_test_split(X_drop_cols, X_impute_median, y, test_size=0.2, random_state=42)
)

In [ ]:
# train logistic regression models
lr_model_cols = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced",  # Handle class imbalance
    solver="lbfgs",
)

lr_model_median = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced",  # Handle class imbalance
    solver="lbfgs",
)

print("Training logistic regression models...")
lr_model_cols.fit(X_cols_train, y_train)
lr_model_median.fit(X_median_train, y_train)
print("Models trained successfully!")

print(f"\nTraining Accuracy (Cols): {lr_model_cols.score(X_cols_train, y_train):.4f}")
print(f"Test Accuracy (Cols): {lr_model_cols.score(X_cols_test, y_test):.4f}")

print(
    f"\nTraining Accuracy (Median): {lr_model_median.score(X_median_train, y_train):.4f}"
)
print(f"Test Accuracy (Median): {lr_model_median.score(X_median_test, y_test):.4f}")